# Q-Optima in Google Colab

**Hybrid quantum-classical optimisation, with honest benchmarks.** Run everything here in your browser, with no installation.

Each problem is solved three ways and checked against the proven best answer:

| Approach | Method |
|---|---|
| Quantum (gate model) | **QAOA** on an exact statevector simulator (optional: real IBM hardware at the end) |
| Quantum-inspired | **Simulated annealing** |
| Classical | Exact solvers and everyday heuristics |

> No quantum speed-up is claimed. At these sizes classical methods win, and the notebook shows it.

**How to use:** click **Runtime → Run all**, or run the cells one by one with **Shift+Enter**.

Repository: [https://github.com/shaswatprof2000-git-hub/q-optima](https://github.com/shaswatprof2000-git-hub/q-optima)

## 1. Get the code

In [ ]:
# Download the repository into Colab (re-running this cell is safe)
import os, sys
if not os.path.exists("q-optima"):
    !git clone -q https://github.com/shaswatprof2000-git-hub/q-optima.git
%cd q-optima
sys.path.insert(0, os.getcwd())   # make `import qoptima` work without a reinstall

import qoptima
print("Q-Optima version", qoptima.__version__)

## 2. Check that everything works
Runs the 11 automated tests. You should see **11 passed**.

In [ ]:
!pip -q install pytest
!python -m pytest -q

## 3. Route planning: best order to visit places

A depot plus 4 stops around Bhubaneswar–Cuttack becomes a **16-qubit** problem. Change the places, or add your own `"name": (latitude, longitude)` (keep it to 5 places in total for the quantum simulator).

In [ ]:
from qoptima.problems import RoutingProblem
from qoptima import solve_annealing, solve_qaoa

places = {
    "Office (Bhubaneswar)": (20.2961, 85.8245),
    "Phulnakhara site":     (20.3900, 85.8600),
    "Cuttack client":       (20.4625, 85.8830),
    "Patia client":         (20.3530, 85.8190),
    "Khandagiri client":    (20.2560, 85.7790),
}   # approximate coordinates; distances = straight line x 1.3 for roads

route = RoutingProblem.from_latlon(places)
qubo = route.to_qubo()
print(f"{route.n} places -> {qubo.n} qubits\n")

best = route.held_karp()
print(f"Classical exact : {route.describe(best.bits)}  ({best.energy:.1f} km)")

sa = solve_annealing(qubo)
tour, km = route.decode(sa.bits)
print(f"Annealing       : {route.describe(tour)}  ({km:.1f} km)")

qa = solve_qaoa(qubo, p=2, restarts=3)      # ~20 s at 16 qubits
tour, km = route.decode(qa.bits)
print(f"QAOA (simulated): {route.describe(tour)}  ({km:.1f} km)")

In [ ]:
import matplotlib.pyplot as plt

BLUE, INK, MUTED = "#2a78d6", "#0b0b0b", "#52514e"
names = list(places)
order = route.decode(qa.bits)[0] or list(best.bits)
lats = [places[names[i]][0] for i in order]
lons = [places[names[i]][1] for i in order]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(lons, lats, color=BLUE, linewidth=2, zorder=1)
ax.scatter(lons[1:-1], lats[1:-1], s=70, color=BLUE, edgecolor="white", linewidth=2, zorder=2)
ax.scatter([lons[0]], [lats[0]], s=140, marker="s", color=INK, edgecolor="white", linewidth=2, zorder=3)
for step, i in enumerate(order[:-1]):
    lon, lat = places[names[i]][1], places[names[i]][0]
    label = names[i] if step == 0 else f"{step}. {names[i]}"
    ax.annotate(label, (lon, lat), xytext=(8, 6), textcoords="offset points", color=INK, fontsize=9)
ax.set_title(f"Best visiting order: {route.tour_length(order):.1f} km", color=INK, loc="left")
ax.set_xlabel("Longitude", color=MUTED); ax.set_ylabel("Latitude", color=MUTED)
ax.tick_params(colors=MUTED)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.grid(alpha=0.15)
plt.show()

### What did the quantum algorithm actually do?

Before training, every one of the 65,536 possible answers is equally likely, and only 24 of them are valid routes. Training QAOA shifts probability towards good answers. The honest picture: the best route becomes roughly **80x more likely than random guessing**, but most measurements are still invalid. That's why we take many measurements (shots) and keep the best valid one.

In [ ]:
import numpy as np
from qoptima.solvers.qaoa import qaoa_state

energies = qubo.all_energies()
cost = (energies - energies.mean()) / energies.std()
psi = qaoa_state(cost, qa.info["gammas"], qa.info["betas"], qubo.n)
probs = np.abs(psi) ** 2
n_states = len(probs)
valid = np.array([route.decode(qubo.index_to_bits(k, qubo.n))[0] is not None for k in range(n_states)])
k_opt = int(np.argmin(energies))

groups = ["Best route", f"Any valid route ({valid.sum()} of {n_states:,})"]
random_pct = [100 / n_states, 100 * valid.sum() / n_states]
qaoa_pct = [100 * probs[k_opt], 100 * probs[valid].sum()]

GRAY = "#b8b7b0"
fig, ax = plt.subplots(figsize=(8, 3.6))
y = np.arange(len(groups))
ax.barh(y + 0.2, random_pct, height=0.36, color=GRAY, label="Random guessing")
ax.barh(y - 0.2, qaoa_pct, height=0.36, color=BLUE, label="Trained QAOA (p=2)")
for yi, r, q in zip(y, random_pct, qaoa_pct):
    ax.text(r * 1.15, yi + 0.2, f"{r:.4f}%", va="center", fontsize=9, color=MUTED)
    ax.text(q * 1.15, yi - 0.2, f"{q:.2f}%  ({q / r:.0f}x)", va="center", fontsize=9, color=INK)
ax.set_xscale("log")
ax.set_xlim(min(random_pct) / 2, max(qaoa_pct) * 20)
ax.set_yticks(y, groups, color=INK)
ax.invert_yaxis()
ax.set_xlabel("Chance of measuring it in one shot (%, log scale)", color=MUTED)
ax.set_title("QAOA amplifies good answers, but most single shots are still invalid",
             color=INK, loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower right")
ax.tick_params(colors=MUTED)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
plt.show()
shots = 1024
print(f"Invalid answers still take {100 * probs[~valid].sum():.1f}% of the probability.")
print(f"In {shots} shots you expect about {shots * probs[valid].sum():.0f} valid routes "
      f"(vs {shots * valid.sum() / n_states:.1f} by random guessing), and keep the shortest.")

## 4. Portfolio selection: pick k stocks, balancing return and risk

Uses **synthetic demo prices** (clearly not real market data). To use your own data, upload a CSV with a `date` column plus one close-price column per stock, and change the file name below.

*Educational only, not investment advice.*

In [ ]:
from qoptima.problems import PortfolioProblem

pf = PortfolioProblem.from_csv("examples/prices_synthetic.csv", k=3)
pq = pf.to_qubo()
print(f"{pf.n} stocks, pick {pf.k} -> {pq.n} qubits\n")

for label, res in [("Classical exact ", pf.brute_force()),
                   ("Greedy shortcut ", pf.greedy()),
                   ("Annealing       ", solve_annealing(pq)),
                   ("QAOA (simulated)", solve_qaoa(pq))]:
    picks, _ = pf.decode(res.bits)
    s = pf.stats(res.bits)
    print(f"{label}: {picks}  expected return {s['expected_return']:.1%}, volatility {s['volatility']:.1%}")

## 5. Raj Sundari plug-in: match buyers to plots

Scores each buyer–plot pair on budget, size, facing, road width and corner preference, then finds the allocation with the best total fit. The data is **illustrative**. Edit `examples/buyers.csv` and `examples/plots.csv` (Colab's file panel on the left) to try your own.

In [ ]:
import pandas as pd
from qoptima.problems import Buyer, Plot, PlotMatchingProblem

bdf, pdf = pd.read_csv("examples/buyers.csv"), pd.read_csv("examples/plots.csv")
display(bdf); display(pdf)

buyers = [Buyer(r["name"], r["budget_lakh"], r["min_area_sqft"], tuple(str(r["preferred_facing"]).split("|")),
                r["min_road_ft"], bool(r["wants_corner"])) for _, r in bdf.iterrows()]
plots = [Plot(r["plot_id"], r["area_sqft"], r["price_lakh"], r["facing"], r["road_width_ft"], bool(r["corner"]))
         for _, r in pdf.iterrows()]
pm = PlotMatchingProblem(buyers, plots)

print("Fit scores (0-100):")
display(pd.DataFrame(pm.score, index=[b.name for b in buyers], columns=[p.plot_id for p in plots]))

mq = pm.to_qubo()
print(f"\n{len(buyers)} buyers x {len(plots)} plots -> {mq.n} qubits")
for label, res in [("Classical (Hungarian)", pm.hungarian()), ("QAOA (simulated)     ", solve_qaoa(mq))]:
    pairs, total = pm.decode(res.bits)
    print(f"{label}: {pairs}  total fit {total:.1f}")

## 6. Full benchmark
Runs all 6 test problems (8–16 qubits) and writes `results/benchmark.md`. Takes about 40–60 seconds.

In [ ]:
from qoptima import benchmark
from IPython.display import Markdown

df = benchmark.run(verbose=False)
display(Markdown(open("results/benchmark.md").read()))

## 7. Optional: run on a real IBM quantum computer

1. Create a free account at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) and copy your API key.
2. In Colab, click the 🔑 **Secrets** icon on the left, add a secret named `IBM_QUANTUM_TOKEN` with your key, and turn on notebook access.
3. Set `RUN_ON_IBM = True` below and run the cell.

The circuit is trained on the simulator, then sampled once on real hardware, so it uses very little machine time. Real devices are noisy, so expect worse answers than the simulator. Measuring *how much worse* is the interesting part.

> This hardware path hasn't been tested end to end yet, and IBM's account-setup options change between library versions. If it errors, check the current `qiskit-ibm-runtime` docs.

In [ ]:
RUN_ON_IBM = False

if RUN_ON_IBM:
    !pip -q install qiskit qiskit-ibm-runtime
    from google.colab import userdata
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qoptima.solvers.qiskit_backend import sample_circuit

    QiskitRuntimeService.save_account(channel="ibm_quantum_platform",
                                      token=userdata.get("IBM_QUANTUM_TOKEN"), overwrite=True)
    small = RoutingProblem.from_latlon(dict(list(places.items())[:4]))   # 9 qubits: shallower circuit
    sq = small.to_qubo()
    trained = solve_qaoa(sq)
    hw = sample_circuit(sq, trained, shots=2000, use_ibm=True)
    tour, km = small.decode(hw.bits)
    print(hw.solver, "->", small.describe(tour) if tour else "infeasible", f"({km:.1f} km)" if km else "")
    print("Classical optimum:", f"{small.held_karp().energy:.1f} km")
else:
    print("Skipped. Set RUN_ON_IBM = True to try real hardware.")

---
Built by Shaswat Mohanty · MIT licence · [https://github.com/shaswatprof2000-git-hub/q-optima](https://github.com/shaswatprof2000-git-hub/q-optima)